In [14]:
# Imports and path setup

import pandas as pd
from pathlib import Path
import sys
sys.path.insert(0, "src")

import importlib
import ingest
import qc_checks
import kpi

importlib.reload(ingest)
importlib.reload(qc_checks)
importlib.reload(kpi)

<module 'kpi' from '/Users/mansi/Documents/Projects/esg-emissions-pipeline/src/kpi.py'>

In [15]:
# Step1: Ingest raw data 
df = ingest.run(
    raw_path="data/raw/owid-co2-data.csv",
    landing_path="data/raw/portfolio_landing.csv",
)
print(f"Ingested {len(df)} rows across {df['country'].nunique()} entities.")
df.head()

Ingested 240 rows across 10 entities.


,country,iso_code,year,population,gdp,co2,co2_per_capita,co2_growth_prct,methane,nitrous_oxide,total_ghg,ghg_per_capita,cumulative_co2,share_global_co2
0,Brazil,BRA,2000,174018278.0,1.739728e+12,340.183,1.955,3.950,417.310,123.375,2823.803,16.227,7522.605,1.333
1,Brazil,BRA,2001,176301201.0,1.785513e+12,346.166,1.963,1.759,429.615,127.265,2783.375,15.788,7868.771,1.347
2,Brazil,BRA,2002,178503485.0,1.861742e+12,347.765,1.948,0.462,450.588,132.379,2981.136,16.701,8216.536,1.324
3,Brazil,BRA,2003,180622688.0,1.904933e+12,344.645,1.908,-0.897,471.538,137.421,3846.776,21.297,8561.181,1.246
4,Brazil,BRA,2004,182675144.0,2.038448e+12,361.434,1.979,4.871,494.991,145.924,3313.758,18.140,8922.615,1.263


In [9]:
# Step2: QA/QC

summary = qc_checks.run(
    landing_path="data/raw/portfolio_landing.csv",
    processed_path="data/processed/portfolio_clean.csv",
    errors_path="data/errors/portfolio_rejects.csv",
)
summary

{'total_records': 240,
 'passed': 240,
 'rejected': 0,
 'completeness_score': np.float64(0.997)}

In [10]:
# Step3: Calculate KPIs

kpi.run(
    processed_path="data/processed/portfolio_clean.csv",
    output_dir="data/processed/kpis",
)

KPIs written to data/processed/kpis (latest year: 2023)


In [11]:
# Inspecting KPI outputs


kpi_dir = Path("data/processed/kpis")

for file in sorted(kpi_dir.glob("*.csv")):
    print(f"--- {file.name} ---")
    display(pd.read_csv(file))

--- kpi_completeness_by_entity.csv ---


,country,completeness_score
0,Brazil,0.9968
1,China,0.9968
2,Denmark,0.9968
3,Germany,0.9968
4,India,0.9968
5,Netherlands,0.9968
6,Norway,0.9968
7,Sweden,0.9968
8,United Kingdom,0.9968
9,United States,0.9968


--- kpi_intensity_gdp.csv ---


,country,year,co2_per_gdp_million
0,Brazil,2000,0.000196
1,Brazil,2001,0.000194
2,Brazil,2002,0.000187
3,Brazil,2003,0.000181
4,Brazil,2004,0.000177
...,...,...,...
235,United States,2019,0.000282
236,United States,2020,0.000260
237,United States,2021,0.000263
238,United States,2022,0.000259


--- kpi_portfolio_ghg_total.csv ---


,year,portfolio_total_ghg
0,2000,19499.367
1,2001,19237.866
2,2002,19786.941
3,2003,21694.385
4,2004,21751.185
5,2005,22384.456
6,2006,22563.556
7,2007,23195.043
8,2008,23628.616
9,2009,23299.480


--- kpi_total_co2_latest.csv ---


,country,co2
0,China,12172.009
1,United States,4918.407
2,India,3062.756
3,Germany,593.766
4,Brazil,483.992
5,United Kingdom,307.826
6,Netherlands,117.016
7,Norway,38.869
8,Sweden,36.709
9,Denmark,28.831


--- kpi_yoy_change.csv ---


,country,year,yoy_change_pct
0,Brazil,2000,NaN
1,Brazil,2001,1.758759
2,Brazil,2002,0.461917
3,Brazil,2003,-0.897158
4,Brazil,2004,4.871389
...,...,...,...
235,United States,2019,-2.337595
236,United States,2020,-10.427181
237,United States,2021,7.039664
238,United States,2022,0.703012
